# Semana 15 — Proyecto 2: Agente de Investigación Autónomo

**Objetivo:** construir `ResearchAgent`, un agente que recibe un tema, planifica su investigación,
busca fuentes en la web, lee y resume el contenido, y entrega un informe estructurado en Markdown.

**Lo que construyes:**
- `proyectos/research-agent/agent/tools.py` — herramientas del agente
- `proyectos/research-agent/agent/planner.py` — planificación con LLM
- `proyectos/research-agent/agent/executor.py` — bucle ReAct por subtarea
- `proyectos/research-agent/agent/synthesizer.py` — síntesis del informe final
- `proyectos/research-agent/agent/agent.py` — clase `ResearchAgent` que orquesta todo
- `proyectos/research-agent/app.py` — CLI para lanzar el agente

**Prerrequisitos:** semanas 9 (ReAct), 10 (memoria), 11 (Plan-and-Execute).

**Coste estimado de este notebook:** ~$0.01–0.05 con `gpt-4o-mini` (3-5 informes de prueba).

## Celda 0 — Setup

In [ ]:
import subprocess, sys

paquetes = ["openai", "httpx", "beautifulsoup4", "python-dotenv"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + paquetes)

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Define OPENAI_API_KEY en tu .env"
print("✓ API key cargada")

# Directorio del proyecto
PROJECT_DIR = Path("../proyectos/research-agent").resolve()
AGENT_DIR   = PROJECT_DIR / "agent"
INFORMES_DIR = PROJECT_DIR / "informes"

for d in [AGENT_DIR, INFORMES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Proyecto en: {PROJECT_DIR}")

## Celda 1 — Herramientas del agente (`tools.py`)

Implementamos cuatro herramientas. Empieza verificando que `buscar_web` y `leer_url` funcionan
antes de encapsularlos en el agente.

In [ ]:
TOOLS_PY = '''
"""Herramientas para ResearchAgent."""
import httpx
from bs4 import BeautifulSoup


def buscar_web(query: str, max_resultados: int = 5) -> list[dict]:
    """
    Busca en DuckDuckGo (sin API key) y devuelve una lista de
    {titulo, url, descripcion}.
    """
    params = {
        "q": query,
        "format": "json",
        "no_html": "1",
        "skip_disambig": "1",
    }
    try:
        r = httpx.get("https://api.duckduckgo.com/", params=params, timeout=10)
        data = r.json()
    except Exception as e:
        return [{"titulo": "Error", "url": "", "descripcion": str(e)}]

    resultados = []
    for item in data.get("RelatedTopics", []):
        if "FirstURL" in item and len(resultados) < max_resultados:
            resultados.append({
                "titulo":      item.get("Text", "")[:120],
                "url":         item["FirstURL"],
                "descripcion": item.get("Text", ""),
            })
    return resultados


def leer_url(url: str, max_chars: int = 4000) -> str:
    """
    Descarga una URL y devuelve el texto limpio (hasta max_chars caracteres).
    Elimina scripts, estilos y navegación.
    """
    if not url or url.startswith("https://duckduckgo.com"):
        return "URL no válida o no accesible directamente."
    try:
        r = httpx.get(
            url, timeout=15, follow_redirects=True,
            headers={"User-Agent": "Mozilla/5.0 (compatible; ResearchBot/1.0)"}
        )
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        texto = soup.get_text(separator=" ", strip=True)
        # Colapsa espacios múltiples
        import re
        texto = re.sub(r"\s+", " ", texto)
        return texto[:max_chars]
    except Exception as e:
        return f"Error al leer {url}: {e}"


def buscar_en_corpus(query: str, corpus_path: str = "./chroma_db") -> str:
    """
    Busca en el corpus local (Chroma) si existe.
    Devuelve texto vacío si no hay corpus disponible.
    """
    try:
        import chromadb
        from openai import OpenAI
        import os

        client = chromadb.PersistentClient(path=corpus_path)
        collections = client.list_collections()
        if not collections:
            return "No hay corpus interno disponible."

        oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        emb = oai.embeddings.create(
            model="text-embedding-3-small", input=query
        ).data[0].embedding

        col = collections[0]
        res = col.query(query_embeddings=[emb], n_results=3)
        docs = res["documents"][0] if res["documents"] else []
        return "\n\n".join(docs) if docs else "Sin resultados en el corpus."
    except ImportError:
        return "chromadb no instalado — corpus interno no disponible."
    except Exception as e:
        return f"Error buscando en corpus: {e}"


# Definiciones para el LLM (formato OpenAI tool use)
TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "buscar_web",
            "description": "Busca en la web usando DuckDuckGo. Devuelve una lista de resultados con título, URL y descripción.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Términos de búsqueda en inglés o español"},
                    "max_resultados": {"type": "integer", "description": "Número máximo de resultados (por defecto 5)", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "leer_url",
            "description": "Descarga y extrae el texto de una URL. Úsala para leer el contenido de una página web encontrada con buscar_web.",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string", "description": "URL completa a leer"},
                    "max_chars": {"type": "integer", "description": "Máximo de caracteres a extraer (por defecto 4000)", "default": 4000},
                },
                "required": ["url"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_en_corpus",
            "description": "Busca en el corpus interno de documentos indexados localmente (si existe).",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Consulta de búsqueda"},
                },
                "required": ["query"],
            },
        },
    },
]
'''

(AGENT_DIR / "__init__.py").write_text("")
(AGENT_DIR / "tools.py").write_text(TOOLS_PY.strip())
print("✓ agent/tools.py generado")

In [ ]:
# Prueba rápida de las herramientas
import sys
sys.path.insert(0, str(PROJECT_DIR))

from agent.tools import buscar_web, leer_url

print("=== buscar_web ===")
resultados = buscar_web("generative AI aquaculture Europe", max_resultados=3)
for r in resultados:
    print(f"  · {r['titulo'][:70]}")
    print(f"    {r['url']}")

print()
if resultados and resultados[0]["url"]:
    print("=== leer_url (primera URL) ===")
    texto = leer_url(resultados[0]["url"], max_chars=500)
    print(f"  {texto[:300]}...")

## Celda 2 — Planificador (`planner.py`)

El planificador convierte un tema en una lista de subtareas de investigación.

In [ ]:
PLANNER_PY = '''
"""Crea un plan de investigación a partir de un tema."""
import json
from openai import OpenAI
import os

MODELO_PLAN = "gpt-4o-mini"

PROMPT_PLANIFICACION = """Eres un investigador experto en vigilancia tecnológica.
Dado el tema de investigación siguiente, crea un plan de investigación estructurado
con 3-5 subtareas concretas y manejables.

Tema: {tema}

Responde SOLO con JSON válido, sin texto adicional:
{{
  "subtareas": [
    {{"id": 1, "descripcion": "...", "query_sugerida": "..."}},
    ...
  ]
}}

Cada subtarea debe ser:
- Específica y accionable (se puede hacer con búsqueda web o lectura de URL)
- Diferente de las demás (cubre un ángulo distinto del tema)
- Con una query_sugerida en inglés para la búsqueda web"""


def crear_plan(tema: str) -> list[dict]:
    """
    Llama al LLM para crear un plan de investigación.
    Devuelve una lista de subtareas: [{id, descripcion, query_sugerida}].
    """
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp = client.chat.completions.create(
        model=MODELO_PLAN,
        messages=[{"role": "user", "content": PROMPT_PLANIFICACION.format(tema=tema)}],
        response_format={"type": "json_object"},
        temperature=0.3,
    )
    data = json.loads(resp.choices[0].message.content)
    return data.get("subtareas", [])
'''

(AGENT_DIR / "planner.py").write_text(PLANNER_PY.strip())
print("✓ agent/planner.py generado")

# Prueba
from agent.planner import crear_plan

TEMA_PRUEBA = "Aplicaciones de inteligencia artificial generativa en acuicultura y pesca en Europa"
print(f"\nTema: {TEMA_PRUEBA}\n")
plan = crear_plan(TEMA_PRUEBA)

print(f"Plan generado ({len(plan)} subtareas):")
for s in plan:
    print(f"  {s['id']}. {s['descripcion']}")
    print(f"     Query: {s['query_sugerida']}")

## Celda 3 — Executor: bucle ReAct por subtarea (`executor.py`)

Cada subtarea del plan se ejecuta con un bucle ReAct que puede llamar a las herramientas
hasta `MAX_PASOS` veces. Al terminar, devuelve los fragmentos recopilados.

In [ ]:
EXECUTOR_PY = '''
"""Ejecuta una subtarea del plan usando el bucle ReAct + tool use."""
import json
import os
from openai import OpenAI
from agent.tools import buscar_web, leer_url, buscar_en_corpus, TOOL_DEFINITIONS

MODELO_EXEC  = "gpt-4o-mini"
MAX_PASOS    = 6

SYSTEM_EXECUTOR = """Eres un investigador autónomo. Tu tarea es recopilar información
sobre un tema específico usando las herramientas disponibles.

Estrategia:
1. Usa buscar_web para encontrar fuentes relevantes
2. Usa leer_url para leer el contenido de las URLs más prometedoras
3. Usa buscar_en_corpus para consultar documentos internos si es relevante
4. Cuando tengas suficiente información, responde con un resumen conciso
   del hallazgo principal (3-5 frases) y las URLs usadas.

Sé eficiente: no leas más de 3 URLs por subtarea."""


def _ejecutar_herramienta(nombre: str, args: dict) -> str:
    """Despacha la llamada a la herramienta correcta."""
    if nombre == "buscar_web":
        resultados = buscar_web(**args)
        return json.dumps(resultados, ensure_ascii=False)
    elif nombre == "leer_url":
        return leer_url(**args)
    elif nombre == "buscar_en_corpus":
        return buscar_en_corpus(**args)
    else:
        return f"Herramienta desconocida: {nombre}"


def ejecutar_subtarea(subtarea: dict) -> dict:
    """
    Ejecuta una subtarea con el bucle ReAct.
    Devuelve {subtarea_id, descripcion, hallazgo, fuentes, n_pasos}.
    """
    client   = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    messages = [
        {"role": "system",  "content": SYSTEM_EXECUTOR},
        {"role": "user",    "content": f"Investiga: {subtarea['descripcion']}\n"
                                       f"Query sugerida: {subtarea.get('query_sugerida', '')}"},
    ]
    fuentes_usadas = []
    n_pasos        = 0

    for _ in range(MAX_PASOS):
        n_pasos += 1
        resp = client.chat.completions.create(
            model=MODELO_EXEC,
            messages=messages,
            tools=TOOL_DEFINITIONS,
            tool_choice="auto",
        )
        msg = resp.choices[0].message
        messages.append(msg)

        # Sin tool calls → el agente terminó
        if not msg.tool_calls:
            break

        # Ejecuta cada tool call
        for tc in msg.tool_calls:
            nombre = tc.function.name
            args   = json.loads(tc.function.arguments)

            try:
                resultado = _ejecutar_herramienta(nombre, args)
            except Exception as e:
                resultado = f"Error en {nombre}: {e}"

            if nombre == "leer_url" and args.get("url"):
                fuentes_usadas.append(args["url"])

            messages.append({
                "role":         "tool",
                "tool_call_id": tc.id,
                "content":      resultado,
            })

    hallazgo = msg.content or "Sin hallazgo concreto."
    return {
        "subtarea_id":  subtarea["id"],
        "descripcion":  subtarea["descripcion"],
        "hallazgo":     hallazgo,
        "fuentes":      list(dict.fromkeys(fuentes_usadas)),  # deduplica
        "n_pasos":      n_pasos,
    }
'''

(AGENT_DIR / "executor.py").write_text(EXECUTOR_PY.strip())
print("✓ agent/executor.py generado")

# Prueba con la primera subtarea del plan generado en la celda anterior
from agent.executor import ejecutar_subtarea

if plan:
    print(f"\nEjecutando subtarea 1: {plan[0]['descripcion']}\n")
    resultado = ejecutar_subtarea(plan[0])

    print(f"Hallazgo ({resultado['n_pasos']} pasos):")
    print(resultado['hallazgo'][:400])
    print(f"\nFuentes usadas: {resultado['fuentes']}")

## Celda 4 — Sintetizador (`synthesizer.py`)

Recibe todos los hallazgos del plan y genera el informe final en Markdown.

In [ ]:
SYNTHESIZER_PY = '''
"""Genera el informe final a partir de los hallazgos recopilados."""
import os
from openai import OpenAI

MODELO_SINTESIS = "gpt-4o-mini"

PROMPT_INFORME = """Eres un analista senior de vigilancia tecnológica.
A partir de los hallazgos de investigación siguientes, genera un informe
profesional y estructurado sobre el tema.

TEMA: {tema}

HALLAZGOS:
{hallazgos}

INSTRUCCIONES:
- Escribe en español, tono analítico y directo
- Estructura: Resumen ejecutivo (3-5 líneas) → Hallazgos por área → Conclusiones → Fuentes
- Sintetiza e interpreta, no copies literalmente
- Señala tendencias clave, actores relevantes y gaps de información
- Cita fuentes con [URL] al final de cada afirmación que las necesite
- Formato Markdown limpio y legible"""


def generar_informe(tema: str, resultados: list[dict]) -> str:
    """
    Genera el informe final.
    resultados: lista de dicts {subtarea_id, descripcion, hallazgo, fuentes}.
    Devuelve el informe como string Markdown.
    """
    # Formatea los hallazgos para el prompt
    partes = []
    for r in resultados:
        fuentes_str = "\n".join(f"  - {f}" for f in r.get("fuentes", []))
        partes.append(
            f"### Subtarea {r['subtarea_id']}: {r['descripcion']}\n"
            f"{r['hallazgo']}\n"
            f"Fuentes:\n{fuentes_str or '  (sin fuentes externas)'}"
        )
    hallazgos_txt = "\n\n".join(partes)

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    resp   = client.chat.completions.create(
        model=MODELO_SINTESIS,
        messages=[{
            "role":    "user",
            "content": PROMPT_INFORME.format(tema=tema, hallazgos=hallazgos_txt),
        }],
        temperature=0.4,
        max_tokens=2000,
    )
    return resp.choices[0].message.content
'''

(AGENT_DIR / "synthesizer.py").write_text(SYNTHESIZER_PY.strip())
print("✓ agent/synthesizer.py generado")

## Celda 5 — Clase principal `ResearchAgent` (`agent.py`)

Orquesta el flujo completo: planificar → ejecutar cada subtarea → sintetizar → guardar.

In [ ]:
AGENT_PY = '''
"""ResearchAgent: agente autónomo de investigación y vigilancia tecnológica."""
import json
import time
from dataclasses import dataclass, field
from pathlib import Path
from datetime import datetime

from agent.planner    import crear_plan
from agent.executor   import ejecutar_subtarea
from agent.synthesizer import generar_informe


@dataclass
class ResultadoInvestigacion:
    tema:           str
    plan:           list[dict]
    resultados:     list[dict]
    informe:        str
    duracion_s:     float
    n_fuentes:      int
    ruta_informe:   str = ""


class ResearchAgent:
    """
    Agente autónomo de investigación.

    Uso:
        agent = ResearchAgent(informes_dir="./informes")
        resultado = agent.investigar("IA generativa en acuicultura")
        print(resultado.informe)
    """

    def __init__(
        self,
        informes_dir: str = "./informes",
        verbose: bool = True,
    ):
        self.informes_dir = Path(informes_dir)
        self.informes_dir.mkdir(parents=True, exist_ok=True)
        self.verbose = verbose

    def _log(self, msg: str):
        if self.verbose:
            print(msg)

    def investigar(self, tema: str) -> ResultadoInvestigacion:
        """
        Ejecuta el ciclo completo de investigación.
        1. Planifica las subtareas
        2. Ejecuta cada subtarea (ReAct + tools)
        3. Sintetiza el informe final
        4. Guarda el informe en disco
        """
        inicio = time.time()
        self._log(f"\n🔍 Iniciando investigación: {tema}\n")

        # Fase 1: Planificación
        self._log("📋 Fase 1/3 — Planificando...")
        plan = crear_plan(tema)
        self._log(f"   Plan: {len(plan)} subtareas")
        for s in plan:
            self._log(f"   {s['id']}. {s['descripcion']}")

        # Fase 2: Ejecución de subtareas
        self._log("\n🔎 Fase 2/3 — Investigando...")
        resultados = []
        for subtarea in plan:
            self._log(f"   Subtarea {subtarea['id']}: {subtarea['descripcion'][:60]}...")
            resultado = ejecutar_subtarea(subtarea)
            resultados.append(resultado)
            self._log(f"   → {resultado['n_pasos']} pasos, {len(resultado['fuentes'])} fuentes")

        # Fase 3: Síntesis
        self._log("\n✍️  Fase 3/3 — Sintetizando informe...")
        informe = generar_informe(tema, resultados)

        duracion_s = time.time() - inicio
        n_fuentes  = sum(len(r["fuentes"]) for r in resultados)

        # Guarda el informe
        fecha         = datetime.now().strftime("%Y%m%d_%H%M")
        nombre_slug   = tema[:40].replace(" ", "_").replace("/", "-")
        ruta_informe  = self.informes_dir / f"{fecha}_{nombre_slug}.md"

        cabecera = (
            f"---\n"
            f"tema: {tema}\n"
            f"fecha: {datetime.now().isoformat()}\n"
            f"n_subtareas: {len(plan)}\n"
            f"n_fuentes: {n_fuentes}\n"
            f"duracion_s: {duracion_s:.1f}\n"
            f"---\n\n"
        )
        ruta_informe.write_text(cabecera + informe, encoding="utf-8")

        self._log(f"\n✅ Informe guardado: {ruta_informe}")
        self._log(f"   Duración: {duracion_s:.1f}s | Fuentes: {n_fuentes}")

        return ResultadoInvestigacion(
            tema=tema,
            plan=plan,
            resultados=resultados,
            informe=informe,
            duracion_s=duracion_s,
            n_fuentes=n_fuentes,
            ruta_informe=str(ruta_informe),
        )
'''

(AGENT_DIR / "agent.py").write_text(AGENT_PY.strip())
print("✓ agent/agent.py generado")

## Celda 6 — Prueba del agente completo

Ejecuta el ciclo completo: plan → investigación → informe.

In [ ]:
import importlib
import agent.agent
importlib.reload(agent.agent)

from agent.agent import ResearchAgent

agente = ResearchAgent(
    informes_dir=str(INFORMES_DIR),
    verbose=True,
)

# Tema de ejemplo directamente relevante para DIH DATAlife
TEMA = "Aplicaciones de inteligencia artificial en la industria alimentaria gallega"

resultado = agente.investigar(TEMA)

In [ ]:
# Muestra el informe generado
from IPython.display import Markdown, display

display(Markdown(resultado.informe))

print(f"\n--- Metadatos ---")
print(f"Subtareas ejecutadas: {len(resultado.plan)}")
print(f"Fuentes consultadas:  {resultado.n_fuentes}")
print(f"Duración:             {resultado.duracion_s:.1f}s")
print(f"Informe guardado en:  {resultado.ruta_informe}")

## Celda 7 — CLI (`app.py`)

In [ ]:
APP_PY = '''
#!/usr/bin/env python3
"""
ResearchAgent CLI.
Uso: python app.py "Tema de investigación"
     python app.py "IA en acuicultura" --informes-dir ./mis-informes
"""
import sys
import argparse
from dotenv import load_dotenv

load_dotenv()

from agent.agent import ResearchAgent


def main():
    parser = argparse.ArgumentParser(description="Agente de investigación autónomo")
    parser.add_argument("tema", help="Tema a investigar")
    parser.add_argument(
        "--informes-dir", default="./informes",
        help="Directorio donde guardar los informes (por defecto: ./informes)"
    )
    parser.add_argument(
        "--quiet", action="store_true",
        help="Suprime los mensajes de progreso"
    )
    args = parser.parse_args()

    agente = ResearchAgent(
        informes_dir=args.informes_dir,
        verbose=not args.quiet,
    )
    resultado = agente.investigar(args.tema)

    print(f"\nInforme: {resultado.ruta_informe}")
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''

REQUIREMENTS = """openai>=1.0.0
httpx>=0.27.0
beautifulsoup4>=4.12.0
python-dotenv>=1.0.0
chromadb>=0.5.0
"""

README = """# ResearchAgent — Agente de Investigación Autónomo

Agente que recibe un tema, planifica su investigación, busca fuentes en la web,
lee y resume el contenido, y entrega un informe estructurado en Markdown.

## Instalación

```bash
cd proyectos/research-agent
pip install -r requirements.txt
```

Crea un `.env`:
```
OPENAI_API_KEY=sk-...
```

## Uso

```bash
python app.py "Aplicaciones de IA en acuicultura en Europa"
python app.py "Tendencias en bioeconomía forestal" --informes-dir ./mis-informes
```

Los informes se guardan en `informes/YYYYMMDD_HHMM_Tema.md`.

## Arquitectura

```
agent/
├── tools.py        # buscar_web, leer_url, buscar_en_corpus
├── planner.py      # crear_plan() con LLM (JSON mode)
├── executor.py     # bucle ReAct por subtarea
├── synthesizer.py  # generar_informe() a partir de hallazgos
└── agent.py        # ResearchAgent: orquesta el flujo completo
```

## Coste aproximado

Con `gpt-4o-mini` y 4 subtareas:
- Por informe: ~$0.005–0.02
- 100 informes/mes: ~$0.50–2.00

## Proyecto 2 — Plan de Formación AI Engineering (Semana 15)
"""

(PROJECT_DIR / "app.py").write_text(APP_PY.strip())
(PROJECT_DIR / "requirements.txt").write_text(REQUIREMENTS)
(PROJECT_DIR / "README.md").write_text(README)

print("✓ app.py generado")
print("✓ requirements.txt generado")
print("✓ README.md generado")
print()
print("Estructura del proyecto:")
import subprocess
subprocess.run(["find", str(PROJECT_DIR), "-type", "f", "-not", "-path", "*/__pycache__/*"], 
               capture_output=False)

## Reto — Añadir replanning cuando una subtarea falla

El executor actual registra el error y devuelve un hallazgo vacío. Un agente robusto debería
replanificar: si una subtarea no produce resultados útiles, generar una subtarea alternativa.

**Implementa** `replanificar_subtarea()` en `planner.py`:

```python
def replanificar_subtarea(tema: str, subtarea_fallida: dict, razon: str) -> dict:
    """
    Genera una subtarea alternativa cuando la original no produjo resultados.
    
    Parámetros:
    - tema: el tema general de investigación
    - subtarea_fallida: la subtarea que no funcionó {id, descripcion, query_sugerida}
    - razon: por qué falló (p.ej. "sin resultados en DuckDuckGo", "URL no accesible")
    
    Devuelve: nueva subtarea con query_sugerida distinta
    """
    # TODO: llama al LLM con JSON mode para generar una subtarea alternativa
    # El prompt debe incluir la subtarea fallida y la razón del fallo
    # para que el LLM no repita el mismo error
    pass
```

Luego modifica `ResearchAgent.investigar()` para que si `resultado['hallazgo']` contiene
"Error" o tiene menos de 50 caracteres, llame a `replanificar_subtarea()` y reintente una vez.

**Pista:** define un umbral: `if len(resultado['hallazgo']) < 100 and resultado['n_pasos'] == 1`

In [ ]:
# Esqueleto del reto
RETO_SNIPPET = '''
# En planner.py — añade:

PROMPT_REPLANNING = """La siguiente subtarea de investigación no produjo resultados útiles.

Tema general: {tema}
Subtarea fallida: {descripcion}
Query probada: {query_sugerida}
Razón del fallo: {razon}

Genera UNA subtarea alternativa que cubra el mismo ángulo pero con una estrategia diferente.
Responde SOLO con JSON:
{{"id": {id}, "descripcion": "...", "query_sugerida": "..."}}"""


def replanificar_subtarea(tema: str, subtarea_fallida: dict, razon: str) -> dict:
    # TODO: implementar
    pass


# En agent.py — modifica el bucle de ejecución:

for subtarea in plan:
    resultado = ejecutar_subtarea(subtarea)
    
    # TODO: si el hallazgo es muy corto o contiene "Error",
    # llama a replanificar_subtarea() y reintenta
    
    resultados.append(resultado)
'''
print(RETO_SNIPPET)

## Resumen de la semana 15

| Componente | Archivo | Patrón reutilizado |
|-----------|---------|-------------------|
| Herramientas | `agent/tools.py` | Tool use (S09) |
| Planificador | `agent/planner.py` | Plan-and-Execute (S11) + JSON mode (S05) |
| Executor | `agent/executor.py` | Bucle ReAct (S09) |
| Sintetizador | `agent/synthesizer.py` | Prompt engineering (S02) |
| Orquestador | `agent/agent.py` | Supervisor pattern (S11) |
| CLI | `app.py` | — |
| Replanning | Reto en `planner.py` | Replanning (S11) |

La semana 16 añade **API REST + tests** a ResearchAgent, replicando el patrón de la semana 13
pero con los endpoints propios de un agente asíncrono de larga duración.